# 5G connected vehicle: analysis of the simulation results

This notebook reads the CSV files written by `VehicleSim/run_all.sh` and makes the tables and charts of the report:

| Report section | Notebook section |
|---|---|
| 7. Experimental Design | 7.1 calibration · 7.2 scenarios and factors · 7.3 ideal handover · 7.4 warm-up · 7.5 replicas and CIs (correlogram, fixed vs random start) |
| 8. Results | 8.1 handover period · 8.2 speed · 8.3 load · 8.4 exponential vs lognormal · 8.5 ideal handover · 8.6 ECDF · 8.7 animation |
| 9. Conclusions | 9.1 summary table |

Every chart is saved in `figures/` as `.png` (for the report) and `.html` (interactive).

## 0. Setup

In [1]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype+notebook_connected"   # charts shown inside the notebook

DATA = "data"          # CSV files written by run_all.sh
OUT = "figures"        # where the charts are saved
os.makedirs(OUT, exist_ok=True)

# fixed parameters (the same values as in omnetpp.ini)
L = 10000.0          # road length [m]
N = 11               # number of base stations
ISD = L / (N - 1)    # inter-site distance [m]
OFFSET = 20.0        # distance of the base stations from the road [m]
ALPHA = 4e-8         # S = ALPHA * d^2 [s/m^2]
BACKHAUL = 0.002     # BS <-> servers [s]
DOWNLINK = 0.001     # BS -> vehicle, handover reply [s]
WARMUP = 80          # [s]

COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#7b4fd6"]   # one colour per line
SYMBOLS = ["circle", "square", "diamond", "triangle-up"]
GREY = "#8a8983"

# start the browser used by kaleido only once (makes saving PNGs much faster)
try:
    import kaleido
    kaleido.start_sync_server(silence_warnings=True)
except Exception:
    pass

### Reading the data

`opp_scavetool` writes one row **per value**. `load_runs()` keeps only:
- rows of type `scalar`: the results of the run (e.g. `endToEndDelay:mean`);
- rows of type `itervar`: the factors of the run (`H`, `V`, `meanT`);

and returns a table with **one row per run**. A run is marked `collapsed` when its queue grows beyond 10,000 packets (in all the other runs it never exceeds about 1,600): its mean delay keeps growing with the length of the run, so it is not a steady-state value. `load_vectors()` reads the recorded vectors (time and value of every sample).

In [2]:
def load_runs(config):
    """Reads data/<config>.csv and returns a table with ONE ROW PER RUN."""
    raw = pd.read_csv(os.path.join(DATA, config + ".csv"), low_memory=False)

    # results: one column per scalar name
    scalars = raw[raw["type"] == "scalar"].copy()
    scalars["value"] = scalars["value"].astype(float)
    runs = scalars.pivot_table(index="run", columns="name", values="value")

    # factors: one column per iteration variable
    itervars = raw[raw["type"] == "itervar"]
    for name in itervars["attrname"].unique():
        rows = itervars[itervars["attrname"] == name].set_index("run")
        runs[name] = rows["attrvalue"].astype(float)

    # number of the replication (0 ... 29)
    reps = raw[(raw["type"] == "runattr") & (raw["attrname"] == "repetition")].set_index("run")
    runs["repetition"] = reps["attrvalue"].astype(int)

    # a run COLLAPSES when its queue explodes: in all the stable runs the queue never holds more than
    # ~1,600 packets, in a collapsing run it reaches tens of thousands (nothing in between)
    runs["collapsed"] = runs["queueLength:max"] > 10000
    return runs.reset_index()


def load_vectors(config, name="endToEndDelay:vector"):
    """Reads the vector <name> of every run: returns a list of (run, times, values)."""
    raw = pd.read_csv(os.path.join(DATA, config + ".csv"), low_memory=False)
    vectors = raw[(raw["type"] == "vector") & (raw["name"] == name)]
    itervars = raw[raw["type"] == "itervar"]
    out = []
    for _, row in vectors.iterrows():
        factors = itervars[itervars["run"] == row["run"]]
        factors = dict(zip(factors["attrname"], factors["attrvalue"].astype(float)))
        times = np.array(row["vectime"].split(), dtype=float)
        values = np.array(row["vecvalue"].split(), dtype=float)
        out.append((factors, times, values))
    return out

### Statistics

Each scenario has **30 independent replications**. With $n = 30$ we use the normal approximation for the 95 % confidence interval of the mean:

$$\bar{x} \pm 1.96\,\frac{s}{\sqrt{n}}$$

`summarize()` applies it to each group of runs with the same factors (same scenario).

In [3]:
def confidence_interval(values):
    """Mean and half-width of the 95% confidence interval (normal approximation, n = 30)."""
    values = np.array(values, dtype=float)
    return values.mean(), 1.96 * values.std(ddof=1) / np.sqrt(len(values))


def summarize(runs, group_by, metrics):
    """One row per scenario: mean and CI half-width (column + '_ci') of each metric."""
    rows = []
    for key, group in runs.groupby(group_by):
        row = dict(zip(group_by, key if isinstance(key, tuple) else (key,)))
        row["replicas"] = len(group)
        row["collapsed"] = int(group["collapsed"].sum())       # replications whose queue never emptied
        for m in metrics:
            row[m], row[m + "_ci"] = confidence_interval(group[m])
        rows.append(row)
    return pd.DataFrame(rows)

### Plotting helpers

- `style()` gives every chart the same look, with a **title** and a **subtitle**.
- `add_line()` draws the mean of a metric with its 95 % CI as error bars (values in ms or %). A scenario where some replications **collapsed** is drawn as a **hollow marker** without CI: its mean is not a steady-state value.
- `save()` writes the chart to `figures/` as `.html` and `.png`.

In [4]:
def style(fig, title, subtitle, height=520):
    fig.update_layout(title=dict(text=title, subtitle=dict(text=subtitle)),
                      template="simple_white", height=height, margin=dict(t=110),
                      font=dict(family="Segoe UI, Arial", size=14),
                      paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb",
                      legend=dict(orientation="h", y=-0.22, x=0.5, xanchor="center"))
    fig.update_xaxes(showgrid=True, gridcolor="#e8e7e3")
    fig.update_yaxes(showgrid=True, gridcolor="#e8e7e3")


def add_line(fig, data, x, metric, label, k, scale=1000, dash="solid", row=None, col=None, show_legend=True):
    """Mean of <metric> (times <scale>: 1000 for ms, 100 for %) with its 95% CI.
    Scenarios where some replications collapsed are drawn as HOLLOW markers, without CI."""
    data = data.sort_values(x)
    stable = data[data["collapsed"] == 0]
    unstable = data[data["collapsed"] > 0]
    fig.add_trace(go.Scatter(
        x=stable[x], y=stable[metric] * scale,
        error_y=dict(array=stable[metric + "_ci"] * scale, width=4),
        mode="lines+markers", name=label, legendgroup=label, showlegend=show_legend,
        line=dict(color=COLORS[k], width=2.2, dash=dash), marker=dict(color=COLORS[k], size=8, symbol=SYMBOLS[k])),
        row=row, col=col)
    if len(unstable) > 0:
        fig.add_trace(go.Scatter(
            x=unstable[x], y=unstable[metric] * scale, mode="markers", legendgroup=label, showlegend=False,
            marker=dict(color="white", size=11, symbol=SYMBOLS[k], line=dict(width=2, color=COLORS[k]))),
            row=row, col=col)


def save(fig, name, width=1100, height=520):
    fig.write_html(os.path.join(OUT, name + ".html"), include_plotlyjs="cdn")
    fig.write_image(os.path.join(OUT, name + ".png"), width=width, height=height, scale=2)

### Load the experiments

`summary` has one row per scenario: distribution × E[T] × V × H = 2 × 3 × 3 × 4 = **72 scenarios** (30 replications each).
`ideal` has the same scenarios with the ideal handover (no H): 2 × 3 × 3 = 18 scenarios.

In [5]:
METRICS = ["endToEndDelay:mean", "waitingTime:mean", "serviceTime:mean", "utilization:timeavg",
           "queueLength:timeavg", "subOptimal:timeavg", "handoverDelay:mean", "handoverCount:count"]

runs = []
for config, dist in [("Exp_Exponential", "Exponential"), ("Exp_Lognormal", "Lognormal")]:
    r = load_runs(config)
    r["dist"] = dist
    runs.append(r)
runs = pd.concat(runs)
runs["T_ms"] = runs["meanT"] * 1000
summary = summarize(runs, ["dist", "T_ms", "V", "H"], METRICS)

ideal_runs = []
for config, dist in [("Exp_Ideal_Exponential", "Exponential"), ("Exp_Ideal_Lognormal", "Lognormal")]:
    r = load_runs(config)
    r["dist"] = dist
    ideal_runs.append(r)
ideal_runs = pd.concat(ideal_runs)
ideal_runs["T_ms"] = ideal_runs["meanT"] * 1000
ideal = summarize(ideal_runs, ["dist", "T_ms", "V"], ["endToEndDelay:mean", "serviceTime:mean", "utilization:timeavg"])

summary.to_csv(os.path.join(OUT, "summary_table.csv"), index=False)
print(len(runs), "runs ->", len(summary), "scenarios;", len(ideal_runs), "ideal-handover runs ->", len(ideal), "scenarios")

2160 runs -> 72 scenarios; 540 ideal-handover runs -> 18 scenarios


---
# 7. Experimental Design

## 7.1 Parameter calibration

Fixed parameters and the service times they give. With ideal association the distance along the road is uniform in $[0, \mathrm{ISD}/2]$, so

$$\mathbb{E}[S] = \alpha\left(\frac{\mathrm{ISD}^2}{12} + y_{\mathrm{off}}^2\right), \qquad S_{\mathrm{edge}} = \alpha\left(\frac{\mathrm{ISD}^2}{4} + y_{\mathrm{off}}^2\right)$$

In [6]:
E_S = ALPHA * (ISD**2 / 12 + OFFSET**2)
S_EDGE = ALPHA * (ISD**2 / 4 + OFFSET**2)

calibration = pd.DataFrame([
    ["L (road length)", f"{L:.0f} m"],
    ["N (base stations)", N],
    ["ISD = L / (N - 1)", f"{ISD:.0f} m"],
    ["distance of the BSs from the road", f"{OFFSET:.0f} m"],
    ["alpha", f"{ALPHA:.0e} s/m^2"],
    ["backhaul delay", f"{BACKHAUL * 1000:.0f} ms"],
    ["downlink delay (handover reply)", f"{DOWNLINK * 1000:.0f} ms"],
    ["E[S] with ideal association", f"{E_S * 1000:.3f} ms"],
    ["S at the cell edge", f"{S_EDGE * 1000:.3f} ms"],
], columns=["Parameter", "Value"])
calibration

,Parameter,Value
0,L (road length),10000 m
1,N (base stations),11
2,ISD = L / (N - 1),1000 m
3,distance of the BSs from the road,20 m
4,alpha,4e-08 s/m^2
5,backhaul delay,2 ms
6,downlink delay (handover reply),1 ms
7,E[S] with ideal association,3.349 ms
8,S at the cell edge,10.016 ms


## 7.2 Scenarios and factors

Full factorial design: every combination of the levels below, for both distributions of $T$.

**Stability check.** Application and handover packets share the transmitter, so its utilisation is

$$\rho = \left(\frac{1}{\mathbb{E}[T]} + \frac{1}{H}\right)\mathbb{E}[S] \quad < 1$$

The table compares this value (ideal association) with the utilisation **measured** in the simulation (mean over V and the two distributions). The measured value is a little higher because a vehicle on a wrong BS has a larger S.

In [7]:
factors = pd.DataFrame([
    ["E[T] (mean inter-arrival time)", "15, 10, 7.5 ms"],
    ["H (handover period)", "0.02, 0.1, 1, 5 s"],
    ["V (speed)", "15, 25, 40 m/s"],
    ["distribution of T", "exponential (CoV 1), lognormal (CoV 2), same mean"],
], columns=["Factor", "Levels"])
display(factors)

stability = summary.groupby(["T_ms", "H"])["utilization:timeavg"].mean().reset_index()
stability["rho (theory)"] = (1 / (stability["T_ms"] / 1000) + 1 / stability["H"]) * E_S
stability = stability.rename(columns={"T_ms": "E[T] [ms]", "utilization:timeavg": "rho (simulation)"})
stability.round(3)

,Factor,Levels
0,E[T] (mean inter-arrival time),"15, 10, 7.5 ms"
1,H (handover period),"0.02, 0.1, 1, 5 s"
2,V (speed),"15, 25, 40 m/s"
3,distribution of T,"exponential (CoV 1), lognormal (CoV 2), same mean"


,E[T] [ms],H,rho (simulation),rho (theory)
0,7.5,0.02,0.553,0.614
1,7.5,0.10,0.464,0.480
2,7.5,1.00,0.443,0.450
3,7.5,5.00,0.462,0.447
4,10.0,0.02,0.481,0.502
5,10.0,0.10,0.367,0.368
6,10.0,1.00,0.339,0.338
7,10.0,5.00,0.367,0.336
8,15.0,0.02,0.389,0.391
9,15.0,0.10,0.257,0.257


## 7.3 Ideal handover lower bound

The same scenarios are run with `idealHandover = true`: no handover packets, and the vehicle always uses the closest BS. It is the best delay reachable with this radio model; the results are compared in section 8.5.

In [8]:
ideal_table = ideal.copy()
ideal_table["E2E delay [ms]"] = (ideal_table["endToEndDelay:mean"] * 1000).round(2)
ideal_table["rho"] = ideal_table["utilization:timeavg"].round(3)
ideal_table.pivot_table(index=["dist", "T_ms"], columns="V", values="E2E delay [ms]")

V                   15.0    25.0   40.0
dist        T_ms                       
Exponential 7.5   174.73  112.69  76.94
            10.0   11.95   11.73  11.25
            15.0    6.72    6.73   6.71
Lognormal   7.5   210.72  140.18  99.90
            10.0   23.81   22.43  20.83
            15.0    8.47    8.45   8.50

## 7.4 Warm-up and simulation length

`Calibration_Warmup`: 20 independent replications of 800 s with **no warm-up**, under a heavy load (lognormal, E[T] = 7.5 ms, CoV = 2, V = 25 m/s, H = 0.1 s).
The delay samples are averaged in windows of **40 s** (the time to cross one cell at 25 m/s), then the windows are averaged over the 20 replications (with a 95 % CI).

If the first windows are not different from the later ones, the initial transient is short. We discard the first **80 s** (two cells) and measure for **2400 s**.

In [9]:
WINDOW = 40   # [s]
warm = load_vectors("Calibration_Warmup")
edges = np.arange(0, 800 + WINDOW, WINDOW)

window_means = []          # one row per replication, one column per window
for factors, times, values in warm:
    means = [values[(times >= a) & (times < b)].mean() for a, b in zip(edges[:-1], edges[1:])]
    window_means.append(means)
window_means = np.array(window_means)

mean = window_means.mean(axis=0) * 1000
ci = 1.96 * window_means.std(axis=0, ddof=1) / np.sqrt(len(warm)) * 1000
centers = (edges[:-1] + edges[1:]) / 2

fig = go.Figure()
fig.add_trace(go.Scatter(x=centers, y=mean, error_y=dict(array=ci, width=4), mode="lines+markers",
                         name="mean of 20 replications", line=dict(color=COLORS[0], width=2.2)))
fig.add_trace(go.Scatter(x=[WARMUP, WARMUP], y=[0, (mean + ci).max() * 1.1], mode="lines",
                         name="end of warm-up (80 s)", line=dict(color=GREY, dash="dash")))
fig.update_xaxes(title="simulation time [s] (40 s windows)")
fig.update_yaxes(title="mean E2E delay in the window [ms]", rangemode="tozero")
style(fig, "Warm-up: delay over time from an empty system",
      "Calibration_Warmup: lognormal E[T] = 7.5 ms, CoV 2, V = 25 m/s, H = 0.1 s; 20 replications, 95 % CI")
save(fig, "07_warmup")
fig.show()

print(f"first window: {mean[0]:.1f} ms, mean of the other windows: {mean[1:].mean():.1f} ms")

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


first window: 227.8 ms, mean of the other windows: 230.8 ms


## 7.5 Replicas and confidence intervals

How precise are the results with 30 replications? The chart shows, for every scenario, the half-width of the 95 % CI of the mean delay **relative to the mean**. Scenarios where some replications collapsed are left out: they have no steady-state mean (section 8.2).

The widest intervals are at H = 5 s: every replication starts at a random position, so the moment of the handover packets with respect to the cell crossings changes from one replication to the next, and so does the time on a wrong BS. More replications would make these intervals narrower.

In [10]:
summary["ci_relative_%"] = summary["endToEndDelay:mean_ci"] / summary["endToEndDelay:mean"] * 100
stable = summary[summary["collapsed"] == 0]
print("scenarios left out (some replications collapsed, see 8.2):")
print(summary[summary["collapsed"] > 0][["dist", "T_ms", "V", "H", "collapsed"]].to_string(index=False))

fig = go.Figure()
for k, dist in enumerate(["Exponential", "Lognormal"]):
    d = stable[stable["dist"] == dist]
    fig.add_trace(go.Box(y=d["ci_relative_%"], name=dist, marker_color=COLORS[k], boxpoints="all", jitter=0.4))
fig.update_yaxes(title="CI half-width / mean [%]", rangemode="tozero")
style(fig, "Precision of the mean delay with 30 replications",
      "one point per scenario without collapsed replications: half-width of the 95 % CI relative to the mean")
save(fig, "07_ci_width")
fig.show()

stable.groupby("dist")["ci_relative_%"].describe()[["min", "50%", "max"]].round(2)

scenarios left out (some replications collapsed, see 8.2):
       dist  T_ms    V   H  collapsed
Exponential   7.5 40.0 5.0          7
  Lognormal   7.5 40.0 5.0          8


C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


,min,50%,max
dist,,,
Exponential,0.25,1.56,41.62
Lognormal,0.42,2.29,38.28


### Why independent replications: correlogram

A confidence interval needs **IID** samples. The delays of consecutive packets of **one run** are not independent: a packet that waits behind a long queue is followed by another one that waits behind almost the same queue. The **correlogram** shows the autocorrelation at lag $k$ (the correlation between sample $i$ and sample $i+k$); values outside the band $\pm 1.96/\sqrt{n}$ are significantly different from 0 (95 %).

- **Left**: the delays of consecutive packets in one run: strongly correlated, even hundreds of packets apart. These samples cannot be used directly for a CI.
- **Right**: the mean delays of the 30 replications of one scenario, in the order of the replication number: they stay inside the band. Each replication uses different seeds, so the replication means are independent, and the CI is computed on them. The fixed-start experiment (x = 0 for every replication) is shown too: its replications are independent as well.

In [11]:
def autocorrelation(x, max_lag):
    """Autocorrelation of the sequence x at lags 1 ... max_lag."""
    x = np.array(x, dtype=float) - np.mean(x)
    return np.array([np.sum(x[:-k] * x[k:]) / np.sum(x * x) for k in range(1, max_lag + 1)])

# left: delays of consecutive packets in ONE run (Exp_ECDF_Exponential, H = 1 s, first replication)
vectors = load_vectors("Exp_ECDF_Exponential")
f, times, values = [v for v in vectors if v[0]["H"] == 1][0]
samples = values[times >= WARMUP]
acf_run = autocorrelation(samples, 300)
band_run = 1.96 / np.sqrt(len(samples))

# right: mean delay of the 30 replications of one scenario, in replication order
fixed = []
for config, dist in [("Exp_Exponential_x_0", "Exponential"), ("Exp_Lognormal_x_0", "Lognormal")]:
    r = load_runs(config)
    r["dist"] = dist
    fixed.append(r)
fixed = pd.concat(fixed)
fixed["T_ms"] = fixed["meanT"] * 1000

def replication_means(data):
    d = data[(data["dist"] == "Exponential") & (data["T_ms"] == 10) & (data["V"] == 25) & (data["H"] == 1)]
    return d.sort_values("repetition")["endToEndDelay:mean"].values

acf_random = autocorrelation(replication_means(runs), 10)
acf_fixed = autocorrelation(replication_means(fixed), 10)
band_rep = 1.96 / np.sqrt(30)

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"consecutive packets of one run (n = {len(samples)})", "the 30 replication means of one scenario"])
fig.add_trace(go.Bar(x=np.arange(1, 301), y=acf_run, marker_color=COLORS[0], name="packets of one run"), row=1, col=1)
fig.add_trace(go.Bar(x=np.arange(1, 11) - 0.2, y=acf_random, width=0.4, marker_color=COLORS[1],
                     name="replications, random start"), row=1, col=2)
fig.add_trace(go.Bar(x=np.arange(1, 11) + 0.2, y=acf_fixed, width=0.4, marker_color=COLORS[2],
                     name="replications, start at x = 0"), row=1, col=2)
for col, band, last in [(1, band_run, 300), (2, band_rep, 10)]:
    for sign in [1, -1]:
        fig.add_trace(go.Scatter(x=[0.5, last + 0.5], y=[sign * band] * 2, mode="lines", showlegend=False,
                                 line=dict(color=GREY, dash="dash")), row=1, col=col)
fig.update_xaxes(title="lag k")
fig.update_yaxes(title="autocorrelation", range=[-1, 1])
style(fig, "Correlogram: samples inside one run against independent replications",
      "exponential T, E[T] = 10 ms, V = 25 m/s, H = 1 s; dashed lines = 95 % band, +-1.96 / sqrt(n)")
save(fig, "07_correlogram", width=1300)
fig.show()

print(f"one run: autocorrelation at lag 1 = {acf_run[0]:.2f}, at lag 100 = {acf_run[99]:.2f} (band +-{band_run:.3f})")
print(f"replications: largest |autocorrelation| = {np.abs(acf_random).max():.2f} (random start), "
      f"{np.abs(acf_fixed).max():.2f} (start at x = 0), band +-{band_rep:.2f}")

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


one run: autocorrelation at lag 1 = 0.99, at lag 100 = 0.54 (band +-0.007)
replications: largest |autocorrelation| = 0.19 (random start), 0.25 (start at x = 0), band +-0.36


### Fixed start (x = 0) against random start

The same 72 scenarios were also run with **every replication starting at x = 0** (configs `Exp_Exponential_x_0`, `Exp_Lognormal_x_0`). Both experiments give independent replications (see the correlogram), but with x = 0 the moment of the handover packets with respect to the cell crossings is **the same in all of them** (section 8.7), so they estimate the delay of one particular start, not of the system in general.

The table compares the two experiments scenario by scenario: the difference is **significant** when the two 95 % confidence intervals do not overlap. The chart shows the scenarios with E[T] = 10 ms (exponential T): solid = random start, dotted = start at x = 0.

In [12]:
summary_fixed = summarize(fixed, ["dist", "T_ms", "V", "H"], METRICS)
both = summary.merge(summary_fixed, on=["dist", "T_ms", "V", "H"], suffixes=(" random", " x=0"))
both["random [ms]"] = both["endToEndDelay:mean random"] * 1000
both["x = 0 [ms]"] = both["endToEndDelay:mean x=0"] * 1000
both["significant"] = (abs(both["endToEndDelay:mean random"] - both["endToEndDelay:mean x=0"])
                       > both["endToEndDelay:mean_ci random"] + both["endToEndDelay:mean_ci x=0"])
both["collapsed (random / x=0)"] = both["collapsed random"].astype(str) + " / " + both["collapsed x=0"].astype(str)

print(both["significant"].sum(), "of", len(both), "scenarios differ significantly")
print(both.groupby("H")["significant"].sum().rename("significant differences per H").to_string())

fig = go.Figure()
for k, v in enumerate([15, 25, 40]):
    d_random = summary[(summary["dist"] == "Exponential") & (summary["T_ms"] == 10) & (summary["V"] == v)]
    d_fixed = summary_fixed[(summary_fixed["dist"] == "Exponential") & (summary_fixed["T_ms"] == 10) & (summary_fixed["V"] == v)]
    add_line(fig, d_random, "H", "endToEndDelay:mean", f"V = {v} m/s, random start", k)
    add_line(fig, d_fixed, "H", "endToEndDelay:mean", f"V = {v} m/s, x = 0", k, dash="dot")
fig.update_xaxes(type="log", title="H [s] (log)")
fig.update_yaxes(type="log", dtick=1, title="mean E2E delay [ms] (log)")
style(fig, "Random start against fixed start (x = 0)",
      "exponential T, E[T] = 10 ms; mean of 30 replications, 95 % CI", height=560)
save(fig, "07_fixed_vs_random_start")
fig.show()

both[["dist", "T_ms", "V", "H", "random [ms]", "x = 0 [ms]", "significant", "collapsed (random / x=0)"]].round(2)

17 of 72 scenarios differ significantly
H
0.02     0
0.10     0
1.00     3
5.00    14


C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


,dist,T_ms,V,H,random [ms],x = 0 [ms],significant,collapsed (random / x=0)
0,Exponential,7.5,15.0,0.02,1226.55,1219.34,False,0 / 0
1,Exponential,7.5,15.0,0.10,314.55,314.58,False,0 / 0
2,Exponential,7.5,15.0,1.00,205.84,205.60,False,0 / 0
3,Exponential,7.5,15.0,5.00,325.68,345.21,True,0 / 0
4,Exponential,7.5,25.0,0.02,748.10,748.19,False,0 / 0
...,...,...,...,...,...,...,...,...
67,Lognormal,15.0,25.0,5.00,12.51,15.45,True,0 / 0
68,Lognormal,15.0,40.0,0.02,36.47,36.31,False,0 / 0
69,Lognormal,15.0,40.0,0.10,9.58,9.59,False,0 / 0
70,Lognormal,15.0,40.0,1.00,8.95,8.84,False,0 / 0


---
# 8. Results

The end-to-end delay of an application packet is

$$D_{ee} = W_q + S + d_{bh}$$

(waiting time in the queue + service time + 2 ms of backhaul): the charts below often show $W_q$ and $S$ separately to see **which part** changes.

## 8.1 Impact of the handover period H

Mean end-to-end delay against H, one line per speed, one panel per load (exponential T).

In [13]:
def delay_vs_h(dist, name):
    loads = [15, 10, 7.5]
    fig = make_subplots(rows=1, cols=3, subplot_titles=[f"E[T] = {t} ms" for t in loads], shared_yaxes=True)
    for c, t in enumerate(loads):
        for k, v in enumerate([15, 25, 40]):
            d = summary[(summary["dist"] == dist) & (summary["T_ms"] == t) & (summary["V"] == v)]
            add_line(fig, d, "H", "endToEndDelay:mean", f"V = {v} m/s", k, row=1, col=c + 1, show_legend=(c == 0))
        fig.update_xaxes(type="log", title="H [s] (log)", row=1, col=c + 1)
    fig.update_yaxes(type="log", dtick=1)                     # delays go from ~7 ms to ~1 s
    fig.update_yaxes(title="mean E2E delay [ms] (log)", row=1, col=1)
    style(fig, f"End-to-end delay against the handover period ({dist.lower()} T)",
          "mean of 30 replications, 95 % CI; one panel per load; hollow marker = some replications collapsed")
    save(fig, name, width=1300)
    return fig

delay_vs_h("Exponential", "08_delay_vs_H_exponential").show()

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


**Where the delay comes from.** Waiting time and service time against H (exponential T, E[T] = 10 ms, V = 25 m/s), stacked with the 2 ms of backhaul.

In [14]:
d = summary[(summary["dist"] == "Exponential") & (summary["T_ms"] == 10) & (summary["V"] == 25)].sort_values("H")
labels = [f"H = {h:g} s" for h in d["H"]]

fig = go.Figure()
fig.add_trace(go.Bar(x=labels, y=d["waitingTime:mean"] * 1000, name="waiting time W", marker_color=COLORS[1]))
fig.add_trace(go.Bar(x=labels, y=d["serviceTime:mean"] * 1000, name="service time S", marker_color=COLORS[0]))
fig.add_trace(go.Bar(x=labels, y=[BACKHAUL * 1000] * len(d), name="backhaul (2 ms)", marker_color=GREY))
fig.update_layout(barmode="stack")
fig.update_yaxes(title="mean delay [ms]")
style(fig, "Decomposition of the end-to-end delay",
      "exponential T, E[T] = 10 ms, V = 25 m/s; mean of 30 replications")
save(fig, "08_delay_decomposition")
fig.show()

d[["H", "endToEndDelay:mean", "waitingTime:mean", "serviceTime:mean", "utilization:timeavg", "subOptimal:timeavg"]].round(4)

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


,H,endToEndDelay:mean,waitingTime:mean,serviceTime:mean,utilization:timeavg,subOptimal:timeavg
16,0.02,0.2606,0.2554,0.0032,0.4813,0.0301
17,0.10,0.0209,0.0155,0.0033,0.3676,0.0050
18,1.00,0.0131,0.0078,0.0034,0.3387,0.0133
19,5.00,0.0544,0.0488,0.0036,0.3587,0.0806


## 8.2 Impact of the vehicle speed V

The faster the vehicle, the more often it crosses a cell boundary, and the longer it can stay on a BS that is no longer the closest one.
The chart shows the **fraction of time on a sub-optimal BS** against H (exponential T, E[T] = 10 ms).

In [15]:
fig = go.Figure()
for k, v in enumerate([15, 25, 40]):
    d = summary[(summary["dist"] == "Exponential") & (summary["T_ms"] == 10) & (summary["V"] == v)]
    add_line(fig, d, "H", "subOptimal:timeavg", f"V = {v} m/s", k, scale=100)
fig.update_xaxes(type="log", title="H [s] (log)")
fig.update_yaxes(title="time on a sub-optimal BS [%]", rangemode="tozero")
style(fig, "Time on a sub-optimal base station",
      "exponential T, E[T] = 10 ms; mean of 30 replications, 95 % CI")
save(fig, "08_suboptimal_vs_H")
fig.show()

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


**Time on the wrong BS after each crossing.** The vehicle crosses a cell boundary every $\mathrm{ISD}/V$ seconds, so the average time it stays on the old BS after a crossing is

$$\text{time per crossing} = \eta_{sub}\cdot\frac{\mathrm{ISD}}{V}$$

After a crossing, the vehicle waits for the next handover packet (up to $H$) and for its round trip ($D_{ho}$). The table compares the measured time per crossing with $H$ and with the **average** handover delay. When $H$ is small, the time per crossing is much longer than $H + D_{ho}$: the crossing happens at the **cell edge**, where S is largest and the queue is longest, so the handover packet that brings the new BS waits much longer than an average one.

In [16]:
d = summary[(summary["dist"] == "Exponential") & (summary["T_ms"] == 10)].copy()
d["time per crossing [s]"] = d["subOptimal:timeavg"] * ISD / d["V"]
d["mean D_ho [s]"] = d["handoverDelay:mean"]
d["H + mean D_ho [s]"] = d["H"] + d["handoverDelay:mean"]
d[["V", "H", "time per crossing [s]", "mean D_ho [s]", "H + mean D_ho [s]"]].round(3)

,V,H,time per crossing [s],mean D_ho [s],H + mean D_ho [s]
12,15.0,0.02,1.934,0.420,0.440
13,15.0,0.10,0.256,0.027,0.127
14,15.0,1.00,0.641,0.016,1.016
15,15.0,5.00,2.788,0.041,5.041
16,25.0,0.02,1.202,0.262,0.282
17,25.0,0.10,0.202,0.023,0.123
18,25.0,1.00,0.531,0.016,1.016
19,25.0,5.00,3.225,0.096,5.096
20,40.0,0.02,0.784,0.171,0.191
21,40.0,0.10,0.177,0.020,0.120


**Congestion collapse.** The scenarios below have replications whose queue exploded (more than 10,000 packets, section 0). The chart shows the mean delay of **each** of the 30 replications of the worst scenario: most of them stay below about 2 s, a few end with a queue of tens of thousands of packets.

This is a loop: when the vehicle is on a far BS, S is large and the queue grows; the handover packets wait in **the same queue**, so the new BS arrives later; the vehicle stays even longer on the far BS, and S grows again. Whether a run escapes from it depends on where the vehicle is when the queue starts to grow, which is why the start position matters here.

In [17]:
display(summary[summary["collapsed"] > 0][["dist", "T_ms", "V", "H", "collapsed", "replicas"]])

fig = go.Figure()
for k, dist in enumerate(["Exponential", "Lognormal"]):
    d = runs[(runs["dist"] == dist) & (runs["T_ms"] == 7.5) & (runs["V"] == 40) & (runs["H"] == 5)]
    fig.add_trace(go.Box(y=d["endToEndDelay:mean"] * 1000, name=dist, marker_color=COLORS[k],
                         boxpoints="all", jitter=0.5, pointpos=0))
fig.update_yaxes(type="log", dtick=1, title="mean E2E delay of the replication [ms] (log)")
style(fig, "Congestion collapse: the 30 replications of one scenario",
      "E[T] = 7.5 ms, V = 40 m/s, H = 5 s; one point per replication")
save(fig, "08_collapse")
fig.show()

,dist,T_ms,V,H,collapsed,replicas
11,Exponential,7.5,40.0,5.0,7,30
47,Lognormal,7.5,40.0,5.0,8,30


C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


## 8.3 Impact of the traffic load E[T]

Mean delay against H for the three loads (exponential T, V = 25 m/s), with the utilisation of the transmitter.

In [18]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["mean E2E delay", "utilisation of the transmitter"])
for k, t in enumerate([15, 10, 7.5]):
    d = summary[(summary["dist"] == "Exponential") & (summary["T_ms"] == t) & (summary["V"] == 25)]
    add_line(fig, d, "H", "endToEndDelay:mean", f"E[T] = {t} ms", k, row=1, col=1)
    add_line(fig, d, "H", "utilization:timeavg", f"E[T] = {t} ms", k, scale=1, row=1, col=2, show_legend=False)
fig.update_xaxes(type="log", title="H [s] (log)")
fig.update_yaxes(type="log", dtick=1, title="delay [ms] (log)", row=1, col=1)
fig.update_yaxes(title="rho", range=[0, 1], row=1, col=2)
style(fig, "Effect of the load", "exponential T, V = 25 m/s; mean of 30 replications, 95 % CI")
save(fig, "08_load", width=1300)
fig.show()

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


## 8.4 Exponential versus lognormal inter-arrival times

Same mean E[T], but the lognormal (CoV = 2) is burstier than the exponential (CoV = 1). The chart repeats 8.1 for the lognormal; the table compares the two distributions scenario by scenario.

In [19]:
delay_vs_h("Lognormal", "08_delay_vs_H_lognormal").show()

compare = summary.pivot_table(index=["T_ms", "V", "H"], columns="dist", values="endToEndDelay:mean") * 1000
compare["lognormal / exponential"] = compare["Lognormal"] / compare["Exponential"]
compare.round(2)

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


dist            Exponential  Lognormal  lognormal / exponential
T_ms V    H                                                    
7.5  15.0 0.02      1226.55    1282.77                     1.05
          0.10       314.55     358.80                     1.14
          1.00       205.84     248.48                     1.21
          5.00       325.68     380.30                     1.17
     25.0 0.02       748.10     787.39                     1.05
          0.10       198.83     232.33                     1.17
          1.00       137.03     170.24                     1.24
          5.00       406.87     467.60                     1.15
     40.0 0.02       476.08     510.26                     1.07
          0.10       132.65     160.56                     1.21
          1.00       109.64     140.78                     1.28
          5.00     67486.83  110028.99                     1.63
10.0 15.0 0.02       417.76     449.33                     1.08
          0.10        24.54      43.68                     1.78
          1.00        13.14      26.45                     2.01
          5.00        25.64      44.27                     1.73
     25.0 0.02       260.57     288.38                     1.11
          0.10        20.86      37.60                     1.80
          1.00        13.12      25.36                     1.93
          5.00        54.35      79.57                     1.46
     40.0 0.02       169.41     194.26                     1.15
          0.10        17.98      32.37                     1.80
          1.00        14.06      26.31                     1.87
          5.00       140.06     180.58                     1.29
15.0 15.0 0.02        43.94      59.67                     1.36
          0.10         7.19       9.55                     1.33
          1.00         6.79       8.61                     1.27
          5.00         7.13       9.54                     1.34
     25.0 0.02        31.92      45.60                     1.43
          0.10         7.20       9.52                     1.32
          1.00         6.81       8.63                     1.27
          5.00         8.39      12.51                     1.49
     40.0 0.02        24.33      36.47                     1.50
          0.10         7.18       9.58                     1.33
          1.00         6.90       8.95                     1.30
          5.00        14.70      24.50                     1.67

**Waiting time against service time.** The distribution of T changes the queue, not the distance: the service time should be the same for both, the waiting time not.

In [20]:
d = summary[(summary["T_ms"] == 10) & (summary["V"] == 25)]
fig = make_subplots(rows=1, cols=2, subplot_titles=["waiting time W", "service time S"], shared_yaxes=False)
for k, dist in enumerate(["Exponential", "Lognormal"]):
    dd = d[d["dist"] == dist]
    add_line(fig, dd, "H", "waitingTime:mean", dist, k, row=1, col=1)
    add_line(fig, dd, "H", "serviceTime:mean", dist, k, row=1, col=2, show_legend=False)
fig.update_xaxes(type="log", title="H [s] (log)")
fig.update_yaxes(title="[ms]", rangemode="tozero")
style(fig, "Exponential vs lognormal: where the difference is", "E[T] = 10 ms, V = 25 m/s; mean of 30 replications, 95 % CI")
save(fig, "08_exp_vs_lognormal", width=1300)
fig.show()

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


## 8.5 Comparison with the ideal handover

The dashed lines are the ideal handover (always the closest BS, no handover packets) for the same speed and load: the gap between a solid and a dashed line of the same colour is the **cost of the handover policy**.

In [21]:
def compare_ideal(dist, name):
    loads = [15, 10, 7.5]
    fig = make_subplots(rows=1, cols=3, subplot_titles=[f"E[T] = {t} ms" for t in loads], shared_yaxes=True)
    for c, t in enumerate(loads):
        for k, v in enumerate([15, 25, 40]):
            d = summary[(summary["dist"] == dist) & (summary["T_ms"] == t) & (summary["V"] == v)]
            add_line(fig, d, "H", "endToEndDelay:mean", f"V = {v} m/s", k, row=1, col=c + 1, show_legend=(c == 0))
            best = ideal[(ideal["dist"] == dist) & (ideal["T_ms"] == t) & (ideal["V"] == v)]["endToEndDelay:mean"].iloc[0]
            fig.add_trace(go.Scatter(x=[0.02, 5], y=[best * 1000] * 2, mode="lines", showlegend=False,
                                     line=dict(color=COLORS[k], dash="dash", width=1.5)), row=1, col=c + 1)
        fig.update_xaxes(type="log", title="H [s] (log)", row=1, col=c + 1)
    fig.update_yaxes(type="log", dtick=1)                     # delays go from ~7 ms to ~1 s
    fig.update_yaxes(title="mean E2E delay [ms] (log)", row=1, col=1)
    style(fig, f"Real handover (solid) against the ideal handover (dashed), {dist.lower()} T",
          "mean of 30 replications, 95 % CI")
    save(fig, name, width=1300)
    return fig

compare_ideal("Exponential", "08_vs_ideal_exponential").show()
compare_ideal("Lognormal", "08_vs_ideal_lognormal").show()

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


In [22]:
gap = summary.merge(ideal[["dist", "T_ms", "V", "endToEndDelay:mean"]], on=["dist", "T_ms", "V"], suffixes=("", "_ideal"))
gap["delay [ms]"] = gap["endToEndDelay:mean"] * 1000
gap["ideal [ms]"] = gap["endToEndDelay:mean_ideal"] * 1000
gap["extra [%]"] = (gap["endToEndDelay:mean"] / gap["endToEndDelay:mean_ideal"] - 1) * 100
gap.pivot_table(index=["dist", "T_ms", "V"], columns="H", values="extra [%]").round(1)

H                        0.02   0.10  1.00      5.00
dist        T_ms V                                  
Exponential 7.5  15.0   602.0   80.0  17.8      86.4
                 25.0   563.8   76.4  21.6     261.0
                 40.0   518.8   72.4  42.5   87613.7
            10.0 15.0  3395.9  105.4  10.0     114.6
                 25.0  2122.3   77.9  11.9     363.5
                 40.0  1405.6   59.8  25.0    1144.8
            15.0 15.0   553.6    7.0   0.9       6.0
                 25.0   374.2    7.0   1.2      24.6
                 40.0   262.5    7.0   2.8     119.0
Lognormal   7.5  15.0   508.8   70.3  17.9      80.5
                 25.0   461.7   65.7  21.5     233.6
                 40.0   410.8   60.7  40.9  110043.4
            10.0 15.0  1787.0   83.4  11.1      85.9
                 25.0  1185.8   67.7  13.1     254.8
                 40.0   832.8   55.4  26.3     767.1
            15.0 15.0   604.8   12.8   1.7      12.7
                 25.0   439.6   12.6   2.2      48.1
                 40.0   329.3   12.7   5.4     188.4

## 8.6 Delay distributions (ECDF)

The mean hides the tail. `Exp_ECDF_*` record **every** delay (5 replications of 800 s, V = 25 m/s, E[T] = 10 ms). The empirical CDF gives, for every value x, the fraction of packets with delay ≤ x.

In [23]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["exponential T", "lognormal T"], shared_yaxes=True)
percentiles = []
for c, (config, dist) in enumerate([("Exp_ECDF_Exponential", "Exponential"), ("Exp_ECDF_Lognormal", "Lognormal")]):
    vectors = load_vectors(config)
    for k, h in enumerate([0.02, 0.1, 1, 5]):
        # all the samples of the 5 replications with this H, after the warm-up
        samples = np.concatenate([v[t >= WARMUP] for f, t, v in vectors if f["H"] == h]) * 1000
        samples.sort()
        ecdf = np.arange(1, len(samples) + 1) / len(samples)
        step = max(1, len(samples) // 2000)          # 2000 points are enough to draw the curve
        fig.add_trace(go.Scatter(x=samples[::step], y=ecdf[::step], mode="lines", name=f"H = {h:g} s",
                                 legendgroup=str(h), showlegend=(c == 0), line=dict(color=COLORS[k], width=2.2)),
                      row=1, col=c + 1)
        percentiles.append([dist, h, np.percentile(samples, 50), np.percentile(samples, 95), np.percentile(samples, 99)])
fig.update_xaxes(type="log", title="E2E delay [ms] (log)")
fig.update_yaxes(title="fraction of packets", row=1, col=1)
style(fig, "Empirical CDF of the end-to-end delay", "V = 25 m/s, E[T] = 10 ms; 5 replications of 800 s, after the 80 s warm-up")
save(fig, "08_ecdf", width=1300)
fig.show()

pd.DataFrame(percentiles, columns=["dist", "H", "median [ms]", "95th [ms]", "99th [ms]"]).round(2)

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


,dist,H,median [ms],95th [ms],99th [ms]
0,Exponential,0.02,4.91,1436.16,1668.55
1,Exponential,0.10,4.78,114.90,287.92
2,Exponential,1.00,4.75,54.65,177.63
3,Exponential,5.00,4.75,366.24,1234.22
4,Lognormal,0.02,5.08,1554.06,1929.45
5,Lognormal,0.10,4.90,220.50,485.62
6,Lognormal,1.00,4.86,127.65,349.71
7,Lognormal,5.00,4.86,484.23,1474.09


## 8.7 Animation of one run, and why the experiments start at a random position

Made by `make_highway_gif.py` from the config `Animation` (V = 25 m/s, H = 5 s, E[T] = 10 ms exponential, **start at x = 0**), for t = 380 … 480 s: the end of the road, the bounce at x = 10 km and the way back.
The dotted line goes from the car to its serving BS: **green** = the closest BS, **red** = not the closest one.

![highway animation](figures/09_highway_animation.gif)

**What the animation shows: the phase of the handover timer.** Starting at x = 0 with V = 25 m/s, the vehicle crosses a cell boundary at t = 20, 60, 100 … s, which are multiples of H = 5 s: a handover packet is generated **exactly** at each crossing, with x exactly on the boundary (500, 1500 … m). The Mobility Server rounds x / ISD = k + 0.5 **up**:
- going forward, up is the **new** BS: the switch happens a few ms after the crossing (green);
- going back, up is the **old** BS: the vehicle keeps the old BS for one more H plus the round trip, 6–8 s (red), and the delay climbs above 1 s.

If every replication started at x = 0, this alignment would be the same in all of them and would bias the results for many (V, H) pairs. For this reason the experiments (`Exp_*` configs) start at a **random position**, `initialPosition = uniform(0m, 10000m)`, different in every replication: the handover packets then fall at a random point between two crossings, as in reality. The verification tests and this animation keep x = 0, so that their expected values are easy to compute. The experiments were also repeated with x = 0 for every replication: section 7.5 compares the two.

---
# 9. Conclusions

## 9.1 Summary table

For every distribution, load and speed: the H with the **lowest mean delay**, that delay, the ideal-handover delay and the gap between them.

In [24]:
rows = []
for (dist, t, v), group in summary.groupby(["dist", "T_ms", "V"]):
    best = group.loc[group["endToEndDelay:mean"].idxmin()]
    best_ideal = ideal[(ideal["dist"] == dist) & (ideal["T_ms"] == t) & (ideal["V"] == v)]["endToEndDelay:mean"].iloc[0]
    rows.append([dist, t, v, best["H"], best["endToEndDelay:mean"] * 1000, best["endToEndDelay:mean_ci"] * 1000,
                 best_ideal * 1000, (best["endToEndDelay:mean"] / best_ideal - 1) * 100])
best_h = pd.DataFrame(rows, columns=["dist", "E[T] [ms]", "V [m/s]", "best H [s]", "delay [ms]", "CI [ms]",
                                     "ideal [ms]", "extra vs ideal [%]"])
best_h.to_csv(os.path.join(OUT, "best_H.csv"), index=False)
best_h.round(2)

,dist,E[T] [ms],V [m/s],best H [s],delay [ms],CI [ms],ideal [ms],extra vs ideal [%]
0,Exponential,7.5,15.0,1.0,205.84,2.50,174.73,17.80
1,Exponential,7.5,25.0,1.0,137.03,4.21,112.69,21.60
2,Exponential,7.5,40.0,1.0,109.64,6.03,76.94,42.50
3,Exponential,10.0,15.0,1.0,13.14,0.25,11.95,9.97
4,Exponential,10.0,25.0,1.0,13.12,0.37,11.73,11.89
5,Exponential,10.0,40.0,1.0,14.06,0.62,11.25,24.99
6,Exponential,15.0,15.0,1.0,6.79,0.02,6.72,0.93
7,Exponential,15.0,25.0,1.0,6.81,0.02,6.73,1.22
8,Exponential,15.0,40.0,1.0,6.90,0.05,6.71,2.78
9,Lognormal,7.5,15.0,1.0,248.48,5.89,210.72,17.92
